In [1]:
model_path_test='/tf/notebooks/tritonClient/models/simple/1/model.savedmodel'
batch_size=1024
D=1024
class iterator(object):

    def __init__(self, B=batch_size, D=1024):
        self.B = B # batch size
        self.D = D # dimension

    def __iter__(self):
        self.i = 0
        return self

    def __next__(self):
        output = np.float16(np.random.uniform(size=(self.B, self.D)))

        return output

In [2]:
import tensorflow as tf
from tensorflow.python.compiler.tensorrt import trt_convert as trt
D=1024
class WrappedModel(tf.Module):
    def __init__(self):
        super(WrappedModel, self).__init__()
        tf.config.optimizer.set_jit(True)
        self.model = tf.keras.Sequential()
        self.model.add(tf.keras.layers.Dense(64, input_shape=(D,)))

        self.model.add(tf.keras.layers.Dense(32))
        self.model.add(tf.keras.layers.Dense(1))
        self.model.compile(optimizer='sgd', loss='mse')
    @tf.function
    def __call__(self, x):
        return self.model(x)
    
model = WrappedModel()
call = model.__call__.get_concrete_function(tf.TensorSpec([None,None], 
                                            tf.float16, name='input0'))


tf.saved_model.save(model, 
                    model_path_test
#                     ,overwrite=True)
                    ,signatures=call)

    
from tensorflow.python.compiler.tensorrt import trt_convert as trt
    
conversion_params = trt.DEFAULT_TRT_CONVERSION_PARAMS._replace(
  precision_mode=trt.TrtPrecisionMode.FP16,
  max_workspace_size_bytes=8000000000,
  minimum_segment_size=3,
  use_calibration=True,
  maximum_cached_engines=1
)


converter = trt.TrtGraphConverterV2(
  input_saved_model_dir=model_path_test,
  conversion_params=conversion_params
)


converter.convert()
converter.save(output_saved_model_dir=f'{model_path_test}')

INFO:tensorflow:Assets written to: /tf/notebooks/tritonClient/models/simple/1/model.savedmodel/assets
INFO:tensorflow:Linked TensorRT version: (7, 2, 2)
INFO:tensorflow:Loaded TensorRT version: (7, 2, 2)
INFO:tensorflow:Could not find TRTEngineOp_000_000 in TF-TRT cache. This can happen if build() is not called, which means TensorRT engines will be built and cached at runtime.
INFO:tensorflow:Assets written to: /tf/notebooks/tritonClient/models/simple/1/model.savedmodel/assets


In [3]:
import argparse
import numpy as np
import os
import sys
from builtins import range
import time
from tqdm import tqdm
import tritonclient.grpc as grpcclient
from tritonclient import utils
import tritonclient.utils.cuda_shared_memory as cudashm

from ctypes import *

FLAGS = None

if __name__ == '__main__':
    parser = argparse.ArgumentParser()
    parser.add_argument('-v',
                        '--verbose',
                        action="store_true",
                        required=False,
                        default=False,
                        help='Enable verbose output')
    parser.add_argument('-u',
                        '--url',
                        type=str,
                        required=False,
                        default='localhost:8001',
                        help='Inference server URL. Default is localhost:8001.')

#     FLAGS = parser.parse_args()
    FLAGS, unknown = parser.parse_known_args()
    print("")

    try:
        triton_client = grpcclient.InferenceServerClient(url=FLAGS.url,
                                                         verbose=FLAGS.verbose)
    except Exception as e:
        print("channel creation failed: " + str(e))
        sys.exit(1)

    # To make sure no shared memory regions are registered with the
    # server.
    triton_client.unregister_system_shared_memory()
    triton_client.unregister_cuda_shared_memory()

    # We use a simple model that takes 2 input tensors of 16 integers
    # each and returns 2 output tensors of 16 integers each. One
    # output tensor is the element-wise sum of the inputs and one
    # output is the element-wise difference.


    
    
    configuration = """
    name: "simple"
    platform: "tensorflow_savedmodel"
    max_batch_size: 1024
    input [
          {
            name: "input0"
            data_type: TYPE_FP16
            dims: [ 1024 ]
          }
        ]
    output [
          {
            name: "output_0"
            data_type: TYPE_FP32
            dims: [ 1 ]
          }
        ]


    """
    with open('/tf/notebooks/tritonClient/models/simple/config.pbtxt', 'w') as file:
        file.write(configuration)
    
    
    
    model_name = "simple"
    model_version = "1"
    !curl -v localhost:8000/v2/models/simple

    # Create the data for the two input tensors. Initialize the first
    # to unique integers and the second to all ones.
    input0_data = np.arange(start=0, stop=1024, dtype=np.float16)
    
    
    
    
    
    
#     input1_data = np.ones(shape=16, dtype=np.int32)

    input_byte_size = input0_data.size * input0_data.itemsize
    output_byte_size = input_byte_size

    # Create Output0 and Output1 in Shared Memory and store shared memory handles
    shm_op0_handle = cudashm.create_shared_memory_region(
        "output0_data", output_byte_size, 0)


    # Register Output0 and Output1 shared memory with Triton Server
    triton_client.register_cuda_shared_memory(
        "output0_data", cudashm.get_raw_handle(shm_op0_handle), 0,
        output_byte_size)


    # Create Input0 and Input1 in Shared Memory and store shared memory handles
    shm_ip0_handle = cudashm.create_shared_memory_region(
        "input0_data", input_byte_size, 0)


    # Put input data values into shared memory
    cudashm.set_shared_memory_region(shm_ip0_handle, [input0_data])


    # Register Input0 and Input1 shared memory with Triton Server
    triton_client.register_cuda_shared_memory(
        "input0_data", cudashm.get_raw_handle(shm_ip0_handle), 0,
        input_byte_size)

    # Set the parameters to use data from shared memory
    inputs = []
    inputs.append(grpcclient.InferInput('input0', [1, 1024], "FP16"))
    inputs[-1].set_shared_memory("input0_data", input_byte_size)



    outputs = []
    outputs.append(grpcclient.InferRequestedOutput('output_0'))
    outputs[-1].set_shared_memory("output0_data", output_byte_size)



    results = triton_client.infer(model_name=model_name,
                                  inputs=inputs,
                                  outputs=outputs,
                                 model_version="1")
    
    from functools import partial
    import sys

    if sys.version_info >= (3, 0):
        import queue
    else:
        import Queue as queue

    class UserData:

        def __init__(self):
            self._completed_requests = queue.Queue()

    def __init__(self):
        self._completed_requests = queue.Queue()
    def completion_callback(user_data, result, error):
        # passing error raise and handling out
        user_data._completed_requests.put((result, error))

    user_data = UserData()

    async_requests = []
    responses = []
    request_count = 1000
    start_time = time.time()
    

    for i in tqdm(range(request_count)):
            async_requests.append(triton_client.async_infer(model_name=model_name,
                                  inputs=inputs,
                                  callback=partial(completion_callback, user_data),
                                  outputs=outputs,
                                 model_version="1"))


    end_time = time.time()
    batch_size = 1024
    print('Average Latency: ~{} seconds'.format((end_time - start_time) / request_count))
    print('Average Throughput: ~{} examples / second'.format(batch_size * request_count / (end_time - start_time)))
    
    
    # Read results from the shared memory.
    output0 = results.get_output("output_0")
    if output0 is not None:
        output0_data = cudashm.get_contents_as_numpy(
            shm_op0_handle, utils.triton_to_np_dtype(output0.datatype),
            output0.shape)
    else:
        print("OUTPUT0 is missing in the response.")
        sys.exit(1)



    for i in range(1):
        print(
            str(input0_data[i]) + " = " +
            str(output0_data[0]))


    print(len(cudashm.allocated_shared_memory_regions()))
    num = len(cudashm.allocated_shared_memory_regions())
    
    
    
    
    assert len(cudashm.allocated_shared_memory_regions()) == num
    cudashm.destroy_shared_memory_region(shm_ip0_handle)

    cudashm.destroy_shared_memory_region(shm_op0_handle)

    print(len(cudashm.allocated_shared_memory_regions()))
    assert len(cudashm.allocated_shared_memory_regions()) == 0

    print('PASS: cudashm')


E1128 17:56:34.919897071  122152 ev_epollex_linux.cc:515]    Error shutting down fd 161. errno: 9
*   Trying ::1:8000...
* TCP_NODELAY set
* connect to ::1 port 8000 failed: Connection refused
*   Trying 127.0.0.1:8000...
* TCP_NODELAY set
* Connected to localhost (127.0.0.1) port 8000 (#0)
> GET /v2/models/simple HTTP/1.1
> Host: localhost:8000
> User-Agent: curl/7.68.0
> Accept: */*
> 
* Mark bundle as not supporting multiuse
< HTTP/1.1 200 OK
< Content-Type: application/json
< Content-Length: 199
< 
* Connection #0 to host localhost left intact
{"name":"simple","versions":["1"],"platform":"tensorflow_savedmodel","inputs":[{"name":"input0","datatype":"FP16","shape":[-1,1024]}],"outputs":[{"name":"output_0","datatype":"FP32","shape":[-1,1]}]}

100%|██████████| 1000/1000 [00:00<00:00, 5380.89it/s]

Average Latency: ~0.00019485020637512206 seconds
Average Throughput: ~5255319.042509064 examples / second
0.0 = [-343.06924]
2
0
PASS: cudashm
